# 0. Install Seqeval

In [ ]:
!git clone https://github.com/chakki-works/seqeval.git /tmp/seqeval_repo
!cp -r /tmp/seqeval_repo/seqeval /usr/local/lib/python3.13/dist-packages/
!rm -rf /tmp/seqeval_repo
!echo "Install Seq Eval completed"

# 1. Clone repository

In [ ]:
!git clone -q https://github.com/ducbao210/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora.git
%cd Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora

In [ ]:
!pwd

# 2. Dependencies

In [ ]:
!pip install -q -r requirements-kaggle.txt

In [ ]:
from kaggle_secrets import UserSecretsClient
from pathlib import Path

# Kaggle Secret HF_TOKEN_NER must have read + write access to ducbao210/NER.
hf_token = UserSecretsClient().get_secret("HF_TOKEN_NER")

# main.py loads both values automatically through python-dotenv.
Path(".env").write_text(
    f"HF_TOKEN={hf_token}\nHF_CHECKPOINT_REPO_ID=ducbao210/NER\n",
    encoding="utf-8",
)
print("Successfully wrote .env configuration")

# 3. Download dataset

In [ ]:
!python download_data.py

# 4. Training models

`--upload_checkpoint` uploads both the best and latest checkpoints to `ducbao210/NER/checkpoints/`.
The local files remain under `results/checkpoints/<model_prefix>_*.pt`.

## 4.1 Use CRF

In [ ]:
# Train all models with CRF and upload both *_best_model.pt and *_latest_model.pt.
# Outputs are stored under results/checkpoints/, results/plots/, results/metrics/, etc.
!python main.py --mode train --model all --use_crf --upload_checkpoint

## 4.2 Do not use CRF

In [ ]:
# Train all models without CRF and upload both *_best_model.pt and *_latest_model.pt.
!python main.py --mode train --model all --upload_checkpoint

# 5. Evaluations

## 5.1 Use CRF


In [ ]:
# Evaluate all CRF models.
# For each model: local results/checkpoints/<prefix>_best_model.pt is preferred;
# if absent, latest is used; if both are absent, HF is downloaded automatically.
!python main.py --mode evaluate --model all --use_crf

## 5.2 Do not use CRF


In [ ]:
# Evaluate all non-CRF models with the same checkpoint resolution logic.
!python main.py --mode evaluate --model all

# 6. Compress results

In [ ]:
# Do not include the private HF token in the output archive.
!rm -f .env
!zip -qr results.zip results/
!echo "Compressed successfully"